# DeepSeek-OCR 2 — synthetic medical PDFs (GPU only)

This notebook runs **only fake medical test PDFs** from the GitHub experiment branch. No paid service or real patient data. The official model requires a compatible NVIDIA GPU. **A free T4 generally cannot run the documented FlashAttention-2/BF16 path**. GPU allocations and free quotas are not guaranteed.

Sources: [upstream model](https://huggingface.co/deepseek-ai/DeepSeek-OCR-2) · [our experimental GitHub branch](https://github.com/overurdream-collab/OCR_deepseak/tree/experiment/deepseek-ocr2-synthetic)

In Colab: Runtime → Change runtime type → choose GPU. Then run cells **in order**. If the preflight fails, do not proceed with the official-path installation.


In [ ]:
# 1. Check NVIDIA GPU compatibility before downloading large model weights.
import platform, shutil, subprocess
import torch
print('Python:', platform.python_version())
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No NVIDIA CUDA GPU found; choose a GPU runtime in Colab first.')
name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU: {name}, compute capability: {major}.{minor}, VRAM: {vram_gb:.1f} GiB')
print('BF16 supported:', torch.cuda.is_bf16_supported())
if major < 8 or not torch.cuda.is_bf16_supported():
    raise RuntimeError('Official FlashAttention-2/BF16 path incompatible with this GPU (e.g. Tesla T4). Do not install; use an A100/L4-class compatible GPU or separately validate an alternate inference path.')
if vram_gb < 16:
    raise RuntimeError('VRAM less than 16 GiB; this experimental setup is not validated here.')
print('GPU preflight PASS. This does not guarantee successful model inference.')


In [ ]:
# 2. Install upstream-tested components and prepare four SYNTHETIC samples.
# Running this cell downloads open-source packages and model requirements; it does not start a paid service.
import os, subprocess, sys
from pathlib import Path
def run(args, cwd=None):
    print('+', ' '.join(map(str,args)))
    subprocess.run(args, cwd=cwd, check=True)
repo_dir = Path('/content/OCR_deepseak')
if not repo_dir.exists():
    run(['git','clone','--single-branch','-b','experiment/deepseek-ocr2-synthetic','https://github.com/overurdream-collab/OCR_deepseak.git',str(repo_dir)])
run([sys.executable,'-m','pip','install','-q','torch==2.6.0','torchvision==0.21.0','torchaudio==2.6.0','--index-url','https://download.pytorch.org/whl/cu118'])
run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','tokenizers==0.20.3','einops','addict','easydict','PyMuPDF>=1.23','Pillow>=10','reportlab>=4.0'])
run([sys.executable,'-m','pip','install','flash-attn==2.7.3','--no-build-isolation'])
run([sys.executable,'scripts/create_synthetic_pdfs.py'],cwd=repo_dir)
print('Synthetic PDFs ready. If package installation reported a restart requirement, restart the runtime and rerun from the first cell.')


In [ ]:
# 3. Execute the REAL DeepSeek-OCR 2 adapter; NO fallback to Tesseract.
import os, sys, subprocess
from pathlib import Path
repo_dir = Path('/content/OCR_deepseak')
env = dict(os.environ, PYTHONPATH=str(repo_dir))
result = subprocess.run([sys.executable,'-m','src.pilot','--engine','deepseek-ocr2','--samples','samples','--output','results/deepseek-ocr2.json'],cwd=repo_dir,env=env)
if result.returncode != 0:
    raise RuntimeError('DeepSeek-OCR 2 inference failed. Do NOT report baseline OCR results as DeepSeek results. Inspect logs above.')


In [ ]:
# 4. Read the real model's raw substring-coverage report.
# This is NOT verified structured medical extraction accuracy.
import json
from pathlib import Path
p=Path('/content/OCR_deepseak/results/deepseek-ocr2.json')
r=json.loads(p.read_text(encoding='utf-8'))
assert r['actual_engine']=='deepseek-ocr2' and r['deepseek_model_executed']
print('Engine:',r['actual_engine'])
print('Raw substring checks:',r['fields_matched'],'/',r['fields_expected'])
for item in r['documents']:
    print(item['filename'], item['field_checks']['matched'], '/', item['field_checks']['expected'])
print('Result JSON:',p)
# Keep this file for review. Do NOT share real patient data; only synthetic results.
